# Predictability of Non-Nearest-TSS Gold Standards

**Question:** The EGL contains genes that are the effector but are *not* the
closest gene to the sentinel variant TSS. Can the feature matrix actually
distinguish these from negatives — i.e., does it have any signal for them?

If non-nearest positives are invisible to the model (all QTL/functional
features null), they are noise that can only hurt training. If they carry
QTL colocalisation or functional signal, they are the most informative
examples in the set.

We split positives into two groups using `distanceSentinelTssNeighbourhood`:
- **Nearest** (`== 1`): effector is the closest gene to TSS
- **Non-nearest** (`== 0`): effector is NOT the closest gene

And compare both against **negatives** (`GSP == 0`).

In [ ]:
import pyspark.sql.functions as f
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 20)

In [ ]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )
    if Path(app_default_credentials).exists():
        return app_default_credentials
    raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

## 1. Three-Group Overview

In [ ]:
fm = session.spark.read.parquet("2603_training_set_full_fm.parquet").cache()

nearest     = fm.filter((f.col("GSP") == 1) & (f.col("distanceSentinelTssNeighbourhood") == 1)).cache()
non_nearest = fm.filter((f.col("GSP") == 1) & (f.col("distanceSentinelTssNeighbourhood") == 0)).cache()
negatives   = fm.filter(f.col("GSP") == 0).cache()

n_near = nearest.count()
n_far  = non_nearest.count()
n_neg  = negatives.count()
n_pos  = n_near + n_far

print(f"Nearest positives    (dist=1): {n_near:>8,}  ({n_near/n_pos*100:.1f}% of positives)")
print(f"Non-nearest positives(dist=0): {n_far:>8,}  ({n_far/n_pos*100:.1f}% of positives)")
print(f"Negatives                    : {n_neg:>8,}")

## 2. Feature Coverage

What fraction of rows in each group have a **non-null** value for each feature?
A feature that is null for nearly all non-nearest positives is useless for
predicting them.

In [ ]:
functional_features = [
    "eQtlColocClppMaximum",
    "eQtlColocH4Maximum",
    "pQtlColocClppMaximum",
    "pQtlColocH4Maximum",
    "sQtlColocClppMaximum",
    "sQtlColocH4Maximum",
    "e2gMean",
    "vepMaximum",
    "vepMean",
    "credibleSetConfidence",
    "distanceSentinelTss",
]

def coverage(df, n, label):
    row = {"group": label, "n": n}
    for feat in functional_features:
        nonnull = df.filter(f.col(feat).isNotNull()).count()
        row[feat] = round(nonnull / n * 100, 1)
    return row

rows = [
    coverage(nearest,     n_near, "nearest (GSP=1, dist=1)"),
    coverage(non_nearest, n_far,  "non-nearest (GSP=1, dist=0)"),
    coverage(negatives,   n_neg,  "negatives (GSP=0)"),
]

cov = pd.DataFrame(rows).set_index("group")
cov

## 3. QTL Colocalisation Signal Distribution

For each group, compare the distribution of `eQtlColocClppMaximum` — the
strongest single-number summary of eQTL evidence. Nulls treated as 0.

In [ ]:
qtl_col = "eQtlColocClppMaximum"
quantiles = [0.25, 0.5, 0.75, 0.9, 0.95]

def qtl_stats(df, n, label):
    filled = df.withColumn(qtl_col, f.coalesce(f.col(qtl_col), f.lit(0.0)))
    qs = filled.approxQuantile(qtl_col, quantiles, 0.001)
    mn = filled.agg(f.mean(qtl_col)).collect()[0][0]
    frac = filled.filter(f.col(qtl_col) > 0).count() / n
    row = {"group": label, "n": n, "mean": round(mn, 4), "frac_>0": round(frac, 3)}
    for q, v in zip(quantiles, qs):
        row[f"p{int(q*100)}"] = round(v, 4)
    return row

qtl_rows = [
    qtl_stats(nearest,     n_near, "nearest"),
    qtl_stats(non_nearest, n_far,  "non-nearest"),
    qtl_stats(negatives,   n_neg,  "negatives"),
]

pd.DataFrame(qtl_rows).set_index("group")

Repeat for all three QTL types to see if pQTL or sQTL compensate where
eQTL is absent.

In [ ]:
qtl_cols = [
    "eQtlColocClppMaximum",
    "pQtlColocClppMaximum",
    "sQtlColocClppMaximum",
]

def frac_nonzero(df, n, col):
    filled = df.withColumn(col, f.coalesce(f.col(col), f.lit(0.0)))
    return round(filled.filter(f.col(col) > 0).count() / n, 3)

rows = []
for grp, df, n in [("nearest", nearest, n_near),
                    ("non-nearest", non_nearest, n_far),
                    ("negatives", negatives, n_neg)]:
    row = {"group": grp, "n": n}
    for col in qtl_cols:
        row[col] = frac_nonzero(df, n, col)
    rows.append(row)

pd.DataFrame(rows).set_index("group")

## 4. How Distinguishable Are Non-Nearest Positives from Negatives?

Compare mean feature values between non-nearest positives and negatives.
A ratio > 1 means the feature is higher in non-nearest positives — i.e.,
the FM has *some* signal to separate them. A ratio ≈ 1 means the group
is indistinguishable from negatives on that feature.

In [ ]:
all_features = [
    "eQtlColocClppMaximum",
    "eQtlColocH4Maximum",
    "pQtlColocClppMaximum",
    "pQtlColocH4Maximum",
    "sQtlColocClppMaximum",
    "sQtlColocH4Maximum",
    "e2gMean",
    "vepMaximum",
    "vepMean",
]

def group_means(df, label):
    agg_exprs = [
        f.mean(f.coalesce(f.col(c), f.lit(0.0))).alias(c)
        for c in all_features
    ]
    row = df.agg(*agg_exprs).collect()[0].asDict()
    row["group"] = label
    return row

means = pd.DataFrame([
    group_means(nearest,     "nearest"),
    group_means(non_nearest, "non-nearest"),
    group_means(negatives,   "negatives"),
]).set_index("group").round(5)

means

In [ ]:
# Fold-change: non-nearest positives vs negatives
eps = 1e-6
ratio = (means.loc["non-nearest"] + eps) / (means.loc["negatives"] + eps)
ratio_df = ratio.rename("non-nearest / negatives").to_frame().sort_values("non-nearest / negatives", ascending=False)
ratio_df

## 5. Signal Availability per Non-Nearest Positive

For each non-nearest positive row, count how many functional feature
categories carry any signal (non-null, > 0). This directly answers:
**what fraction of non-nearest positives are completely dark to the model?**

In [ ]:
signal_cols = {
    "eQTL": ["eQtlColocClppMaximum", "eQtlColocH4Maximum"],
    "pQTL": ["pQtlColocClppMaximum", "pQtlColocH4Maximum"],
    "sQTL": ["sQtlColocClppMaximum", "sQtlColocH4Maximum"],
    "E2G":  ["e2gMean"],
    "VEP":  ["vepMaximum"],
}

# For each category, flag row as 1 if any feature in category is > 0
nn = non_nearest
for cat, cols in signal_cols.items():
    has_signal = f.greatest(*[f.coalesce(f.col(c), f.lit(0.0)) for c in cols]) > 0
    nn = nn.withColumn(f"has_{cat}", f.when(has_signal, 1).otherwise(0))

# Count number of signal categories available per row
cat_cols = [f"has_{cat}" for cat in signal_cols]
nn = nn.withColumn("n_signal_types", sum(f.col(c) for c in cat_cols))

dist = nn.groupBy("n_signal_types").count().orderBy("n_signal_types")
dist.show()

dark = nn.filter(f.col("n_signal_types") == 0).count()
print(f"\nRows with NO functional signal at all: {dark:,} / {n_far:,}  ({dark/n_far*100:.1f}%)")

### Which signal types are present?

In [ ]:
cat_stats = {}
for cat in signal_cols:
    col = f"has_{cat}"
    cnt = nn.filter(f.col(col) == 1).count()
    cat_stats[cat] = {"n_with_signal": cnt, "pct": round(cnt / n_far * 100, 1)}

pd.DataFrame(cat_stats).T.sort_values("pct", ascending=False)

## 6. Distance Profile

Confirm non-nearest positives are genuinely farther from TSS, and check
whether they are spread across the full range or cluster at intermediate
distances.

In [ ]:
dist_col = "distanceSentinelTss"
qs = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
labels = ["min", "p10", "p25", "p50", "p75", "p90", "max"]

rows = []
for grp, df, n in [("nearest", nearest, n_near),
                    ("non-nearest", non_nearest, n_far),
                    ("negatives (sample 50k)", negatives.sample(False, min(1.0, 50000/n_neg), seed=42), 50000)]:
    qv = df.approxQuantile(dist_col, qs, 0.001)
    row = {"group": grp}
    for lbl, v in zip(labels, qv):
        row[lbl] = int(v) if v is not None else None
    rows.append(row)

pd.DataFrame(rows).set_index("group")

## Summary

| Finding | Implication |
|---------|-------------|
| Non-nearest positives make up ~X% of positives | They are a substantial minority |
| eQTL coverage for non-nearest: X% vs Y% for nearest | Whether QTL evidence compensates for distance |
| X% of non-nearest have NO functional signal | These rows are indistinguishable from negatives |
| Non-nearest / negatives fold-change on eQTL: X | Quantifies the FM's ability to separate them |

**Interpretation:** if a large fraction of non-nearest positives carry zero QTL
or functional signal, they act as **noisy positive labels** — the model sees
them as indistinguishable from negatives, which can degrade precision. These
cases may warrant removal from the training set, or a separate analysis of
whether their EGL source (ChEMBL vs rare variants) explains the signal gap.